In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# OpenQASM 2 e il Qiskit SDK

<details>
<summary><b>Versioni dei pacchetti</b></summary>

Il codice in questa pagina è stato sviluppato utilizzando i seguenti requisiti.
Si consiglia di usare queste versioni o versioni più recenti.

```
qiskit[all]~=2.3.0
```
</details>

Il Qiskit SDK fornisce alcuni strumenti per convertire tra le rappresentazioni OpenQASM dei programmi quantistici e la classe [QuantumCircuit](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit).

<span id="qasm2-import"></span>
## Importare un programma OpenQASM 2 in Qiskit
Due funzioni consentono di importare programmi OpenQASM 2 in Qiskit.
Queste sono [`qasm2.load()`](../api/qiskit/qasm2#load), che accetta un nome di file, e [`qasm2.loads()`](../api/qiskit/qasm2#loads), che accetta il programma OpenQASM 2 come stringa.

In [1]:
import qiskit.qasm2

program = """
    OPENQASM 2.0;
    include "qelib1.inc";
    qreg q[2];
    creg c[2];

    h q[0];
    cx q[0], q[1];

    measure q -> c;
"""
circuit = qiskit.qasm2.loads(program)
circuit.draw()

     ┌───┐     ┌─┐   
q_0: ┤ H ├──■──┤M├───
     └───┘┌─┴─┐└╥┘┌─┐
q_1: ─────┤ X ├─╫─┤M├
          └───┘ ║ └╥┘
c: 2/═══════════╩══╩═
                0  1 

Per ulteriori informazioni, consulta l'[API Qiskit per OpenQASM 2](https://docs.quantum.ibm.com/api/qiskit/qasm2).

### Importare programmi semplici
Per la maggior parte dei programmi OpenQASM 2, puoi usare semplicemente `qasm2.load` e `qasm2.loads` con un singolo argomento.

#### Esempio: importare un programma OpenQASM 2 come stringa
Usa `qasm2.loads()` per importare un programma OpenQASM 2 come stringa in un QuantumCircuit:

In [2]:
from qiskit import qasm2

program = """
    OPENQASM 2.0;
    include "qelib1.inc";

    qreg q[4];
    creg c[4];

    h q[0];
    cx q[0], q[1];

    // 'rxx' is not actually in `qelib1.inc`,
    // but Qiskit used to behave as if it were.
    rxx(0.75) q[2], q[3];

    measure q -> c;
"""
circuit = qasm2.loads(
    program,
    custom_instructions=qasm2.LEGACY_CUSTOM_INSTRUCTIONS,
)

#### Example: use a particular gate class when importing an OpenQASM 2 program

Qiskit cannot, in general, verify if the definition in an OpenQASM 2 `gate` statement corresponds exactly to a Qiskit standard-library gate.
Instead, Qiskit chooses a custom gate using the precise definition supplied.
This can be less efficient that using one of the built-in standard gates, or a user-defined custom gate.
You can manually define `gate` statements with particular classes.

In [3]:
from qiskit import qasm2
from qiskit.circuit import Gate
from qiskit.circuit.library import RZXGate


# Define a custom gate that takes one qubit and two angles.
class MyGate(Gate):
    def __init__(self, theta, phi):
        super().__init__("my", 1, [theta, phi])


custom_instructions = [
    # Link the OpenQASM 2 name 'my' with our custom gate.
    qasm2.CustomInstruction("my", 2, 1, MyGate),
    # Link the OpenQASM 2 name 'rzx' with Qiskit's
    # built-in RZXGate.
    qasm2.CustomInstruction("rzx", 1, 2, RZXGate),
]

program = """
    OPENQASM 2.0;

    gate my(theta, phi) q {
        U(theta / 2, phi, -theta / 2) q;
    }
    gate rzx(theta) a, b {
        // It doesn't matter what definition is
        // supplied, if the parameters match;
        // Qiskit will still use `RZXGate`.
    }

    qreg q[2];
    my(0.25, 0.125) q[0];
    rzx(pi) q[0], q[1];
"""

circuit = qasm2.loads(
    program,
    custom_instructions=custom_instructions,
)

#### Esempio: importare un programma OpenQASM 2 da file
Usa `load()` per importare un programma OpenQASM 2 da file in un QuantumCircuit:

In [4]:
from qiskit import qasm2
from qiskit.circuit import Gate


# Define a custom gate that takes one qubit and two angles.
class MyGate(Gate):
    def __init__(self, theta, phi):
        super().__init__("my", 1, [theta, phi])


custom_instructions = [
    qasm2.CustomInstruction("my", 2, 1, MyGate, builtin=True),
]

program = """
    OPENQASM 2.0;
    qreg q[1];

    my(0.25, 0.125) q[0];
"""

circuit = qasm2.loads(
    program,
    custom_instructions=custom_instructions,
)

<span id="custom-instructions"></span>
### Collegare i gate OpenQASM 2 ai gate di Qiskit
Per impostazione predefinita, l'importatore OpenQASM 2 di Qiskit tratta il file di inclusione `"qelib1.inc"` come una libreria standard *de facto*.
L'importatore considera questo file come contenente esattamente i gate descritti nel [documento originale che definisce OpenQASM 2](https://arxiv.org/abs/1707.03429).
Qiskit utilizzerà i gate integrati nella [circuit library](../api/qiskit/circuit_library) per rappresentare i gate presenti in `"qelib1.inc"`.
I gate definiti nel programma tramite istruzioni OpenQASM 2 `gate` manuali saranno, per impostazione predefinita, costruiti come sottoclassi personalizzate di [Qiskit `Gate`](../api/qiskit/qiskit.circuit.Gate).

Puoi indicare all'importatore di usare classi [`Gate`](../api/qiskit/qiskit.circuit.Gate) specifiche per le istruzioni `gate` che incontra.
Puoi usare questo meccanismo anche per trattare ulteriori nomi di gate come "integrati", ovvero che non richiedono una definizione esplicita.
Se specifichi quali classi di gate usare per le istruzioni `gate` al di fuori di `"qelib1.inc"`, il circuito risultante sarà in genere più efficiente da utilizzare.

> **Warning:** A partire da Qiskit SDK v1.0, l'*esportatore* OpenQASM 2 di Qiskit (vedi [Esportare un circuito Qiskit in OpenQASM 2](#qasm2-export)) si comporta ancora come se `"qelib1.inc"` contenesse più gate di quanti ne contenga effettivamente.
> Ciò significa che le impostazioni predefinite dell'importatore potrebbero non essere in grado di importare un programma esportato dal nostro esportatore.
> Consulta [l'esempio specifico su come lavorare con l'esportatore legacy](#qasm2-import-legacy) per risolvere questo problema.
> 
> Questa discrepanza è un comportamento legacy di Qiskit, e [sarà risolta in una versione successiva di Qiskit](https://github.com/Qiskit/qiskit/issues/10737).

Per passare informazioni su un'istruzione personalizzata all'importatore OpenQASM 2, usa [la classe `qasm2.CustomInstruction`](../api/qiskit/qasm2#qiskit.qasm2.CustomInstruction).
Questa richiede quattro informazioni obbligatorie, nell'ordine:

* Il **nome** del gate, utilizzato nel programma OpenQASM 2
* Il **numero di parametri angolari** che il gate accetta
* Il **numero di qubit** su cui il gate agisce
* La classe o funzione Python **costruttore** per il gate, che accetta i parametri del gate (ma non i qubit) come argomenti individuali

Se l'importatore incontra una definizione `gate` che corrisponde a un'istruzione personalizzata specificata, utilizzerà quelle informazioni personalizzate per ricostruire l'oggetto gate.
Se viene incontrata un'istruzione `gate` che corrisponde al `name` di un'istruzione personalizzata, ma non corrisponde né al numero di parametri né al numero di qubit, l'importatore genererà un [`QASM2ParseError`](../api/qiskit/qasm2#qasm2parseerror), per indicare la discordanza tra le informazioni fornite e il programma.

Inoltre, un quinto argomento `builtin` può essere impostato facoltativamente su `True` per rendere il gate automaticamente disponibile nel programma OpenQASM 2, anche se non è definito esplicitamente.
Se l'importatore incontra una definizione `gate` esplicita per un'istruzione personalizzata integrata, la accetterà silenziosamente.
Come prima, se una definizione esplicita con lo stesso nome non è compatibile con l'istruzione personalizzata fornita, verrà generato un [`QASM2ParseError`](../api/qiskit/qasm2#qasm2parseerror).
Questo è utile per la compatibilità con esportatori OpenQASM 2 più vecchi e con alcune altre piattaforme quantistiche che trattano i "gate base" del proprio hardware come istruzioni integrate.

Qiskit fornisce un attributo dati per lavorare con programmi OpenQASM 2 prodotti da versioni legacy delle [capacità di esportazione OpenQASM 2 di Qiskit](#qasm2-export).
Questo è [`qasm2.LEGACY_CUSTOM_INSTRUCTIONS`](../api/qiskit/qasm2#legacy-compatibility), che può essere passato come argomento `custom_instructions` a [`qasm2.load()`](../api/qiskit/qasm2#load) e [`qasm2.loads()`](../api/qiskit/qasm2#loads).

<span id="qasm2-import-legacy"></span>
#### Esempio: importare un programma creato dall'esportatore legacy di Qiskit
Questo programma OpenQASM 2 usa gate non presenti nella versione originale di `"qelib1.inc"` senza dichiararli, ma che sono gate standard nella libreria di Qiskit.
Puoi usare [`qasm2.LEGACY_CUSTOM_INSTRUCTIONS`](../api/qiskit/qasm2#legacy-compatibility) per indicare facilmente all'importatore di usare lo stesso insieme di gate che l'esportatore OpenQASM 2 di Qiskit utilizzava in precedenza.

In [5]:
import math
from qiskit import qasm2

program = """
    include "qelib1.inc";
    qreg q[2];
    rx(arctan(pi, 3 + add_one(0.2))) q[0];
    cx q[0], q[1];
"""


def add_one(x):
    return x + 1


customs = [
    # Our `add_one` takes only one parameter.
    qasm2.CustomClassical("add_one", 1, add_one),
    # `arctan` takes two parameters, and `math.atan2` implements it.
    qasm2.CustomClassical("arctan", 2, math.atan2),
]
circuit = qasm2.loads(program, custom_classical=customs)

#### Esempio: usare una classe gate specifica nell'importazione di un programma OpenQASM 2
Qiskit non può, in generale, verificare se la definizione in un'istruzione `gate` di OpenQASM 2 corrisponde esattamente a un gate della libreria standard di Qiskit.
Invece, Qiskit sceglie un gate personalizzato usando la definizione precisa fornita.
Questo può essere meno efficiente rispetto all'uso di uno dei gate standard integrati o di un gate personalizzato definito dall'utente.
Puoi definire manualmente le istruzioni `gate` con classi particolari.

In [6]:
from qiskit import QuantumCircuit, qasm2

# Define any circuit.
circuit = QuantumCircuit(2, 2)
circuit.h(0)
circuit.cx(0, 1)
circuit.measure([0, 1], [0, 1])

# Export to a string.
program = qasm2.dumps(circuit)

# Export to a file.
qasm2.dump(circuit, "my_file.qasm")

#### Esempio: definire un nuovo gate integrato in un programma OpenQASM 2
Se l'argomento `builtin=True` è impostato, un gate personalizzato non necessita di una definizione associata.